<a href="https://colab.research.google.com/github/awaisfarooqchaudhry/IB9AU-GenerativeAI-2026/blob/main/task12_gan_floorplans_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task 12 — GAN for Synthetic Floorplan Generation (Google Colab)

This notebook is a **complete beginner-friendly Colab notebook** for **Required Task 12**.

It is adapted from your uploaded reference notebook **`SD3_GAN_Illustration.ipynb`**, but rewritten for the actual assignment task: **train a GAN on floorplan images and generate new synthetic floorplans**.

## What Task 12 asks you to do
- use the notebook **`SD3_GAN_Illustration.ipynb`** as context
- download the zip file **`floorplans_v2-20251223T170650Z-3-001.zip`**
- train a **GAN model** on the floorplan images
- generate new **synthetic floorplans**

## What this notebook does
1. uploads or locates the floorplans zip
2. extracts the images
3. builds a PyTorch dataset
4. trains a **DCGAN-style** generator and discriminator
5. plots generator and discriminator loss
6. generates new synthetic floorplans
7. saves the trained model and generated images

## Colab recommendation
Use **Runtime → Change runtime type → T4 GPU** before training.

## Beginner note
Run the cells **from top to bottom** in order.
Do not skip cells.


## Why this notebook is different from the reference notebook

Your uploaded reference notebook trains a **very simple GAN on MNIST digits** using fully connected layers.

That is fine for 28×28 handwritten digits, but floorplans are real images and need a more image-friendly GAN.

So here we use a **DCGAN-style architecture**:
- convolutional discriminator
- transposed-convolution generator
- resized floorplan images
- grayscale mode for faster Colab training

This still follows the same GAN idea as the reference notebook:
- the **Generator** creates fake images from random noise
- the **Discriminator** tries to detect whether an image is real or fake
- both models improve through adversarial training


## Step 0 — Install packages


In [ ]:
!pip -q install pillow tqdm matplotlib

## Step 1 — Imports and setup


In [ ]:
import os
import zipfile
import random
import math
import shutil
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.utils import make_grid, save_image
from tqdm.auto import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
if device.type == "cuda":
    print("GPU name:", torch.cuda.get_device_name(0))

## Step 2 — Set seed for reproducibility


In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)
print("✅ Seed set.")

## Step 3 — Configuration

These defaults are chosen to be **Colab-friendly**.

### Good default setup for T4
- `IMG_SIZE = 64`
- `BATCH_SIZE = 32`
- `EPOCHS = 15`

### If you want faster training
Reduce:
- `EPOCHS`
- `MAX_IMAGES`

### If Colab runs out of memory
Reduce:
- `BATCH_SIZE`
- `GEN_FEATURES`
- `DISC_FEATURES`


In [ ]:
CONFIG = {
    "IMG_SIZE": 64,
    "CHANNELS": 1,              # grayscale for speed
    "LATENT_DIM": 100,
    "BATCH_SIZE": 32,
    "EPOCHS": 15,
    "LR": 2e-4,
    "BETA1": 0.5,
    "GEN_FEATURES": 64,
    "DISC_FEATURES": 64,
    "NUM_WORKERS": 2,
    "MAX_IMAGES": None,         # set e.g. 2000 for a faster first run
    "PREVIEW_EVERY": 5,
    "GENERATE_N": 16,
    "OUTPUT_DIR": "task12_outputs"
}

COLOR_MODE = "L" if CONFIG["CHANNELS"] == 1 else "RGB"

Path(CONFIG["OUTPUT_DIR"]).mkdir(exist_ok=True)
print(CONFIG)

## Step 4 — Upload the floorplans zip

### Upload this file
`floorplans_v2-20251223T170650Z-3-001.zip`

If it is already present in `/content`, you can skip the upload cell.


In [ ]:
ZIP_PATH = "/content/floorplans_v2-20251223T170650Z-3-001.zip"

if os.path.exists(ZIP_PATH):
    print("✅ Found zip already at:", ZIP_PATH)
else:
    print("Zip not found at the default path.")
    print("Run the next upload cell if needed.")

In [ ]:
# Run this only if the zip is not already in /content
from google.colab import files

uploaded = files.upload()

if len(uploaded) > 0:
    uploaded_name = list(uploaded.keys())[0]
    ZIP_PATH = f"/content/{uploaded_name}"
    print("✅ Uploaded zip path:", ZIP_PATH)
else:
    print("No file uploaded in this run.")

## Step 5 — Extract the zip


In [ ]:
EXTRACT_DIR = "/content/floorplan_data_gan"

if os.path.exists(EXTRACT_DIR):
    shutil.rmtree(EXTRACT_DIR)

os.makedirs(EXTRACT_DIR, exist_ok=True)

if not os.path.exists(ZIP_PATH):
    raise FileNotFoundError(f"Could not find zip file at: {ZIP_PATH}")

with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    zf.extractall(EXTRACT_DIR)

print("✅ Extracted to:", EXTRACT_DIR)

## Step 6 — Find image files recursively


In [ ]:
def find_image_files(root_dir):
    image_extensions = {".png", ".jpg", ".jpeg", ".bmp", ".webp"}
    image_paths = []

    for current_root, _, files in os.walk(root_dir):
        for fname in files:
            ext = Path(fname).suffix.lower()
            if ext in image_extensions:
                image_paths.append(os.path.join(current_root, fname))

    image_paths = sorted(image_paths)
    return image_paths

all_image_paths = find_image_files(EXTRACT_DIR)

if CONFIG["MAX_IMAGES"] is not None:
    all_image_paths = all_image_paths[:CONFIG["MAX_IMAGES"]]

print("Number of floorplan images found:", len(all_image_paths))

if len(all_image_paths) == 0:
    raise RuntimeError("No image files were found after extraction. Check the zip contents.")

## Step 7 — Preview a few real floorplans


In [ ]:
def show_image_grid(image_paths, n=6, cols=3, title="Sample real floorplans"):
    n = min(n, len(image_paths))
    rows = math.ceil(n / cols)

    plt.figure(figsize=(4 * cols, 4 * rows))
    for i in range(n):
        img = Image.open(image_paths[i]).convert(COLOR_MODE)
        plt.subplot(rows, cols, i + 1)
        if CONFIG["CHANNELS"] == 1:
            plt.imshow(img, cmap="gray")
        else:
            plt.imshow(img)
        plt.title(Path(image_paths[i]).name[:30])
        plt.axis("off")
    plt.suptitle(title)
    plt.tight_layout()
    plt.show()

show_image_grid(all_image_paths, n=6, cols=3)

## Step 8 — Create the dataset and dataloader

We convert images to:
- grayscale
- fixed size
- tensors normalized to `[-1, 1]`

That normalization is important because the Generator uses `Tanh()` at the output.


In [ ]:
class FloorplanDataset(Dataset):
    def __init__(self, image_paths, img_size=64, channels=1):
        self.image_paths = image_paths
        self.channels = channels
        self.color_mode = "L" if channels == 1 else "RGB"

        self.transform = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.5] * channels,
                std=[0.5] * channels
            )
        ])

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert(self.color_mode)
        image = self.transform(image)
        return image

dataset = FloorplanDataset(
    image_paths=all_image_paths,
    img_size=CONFIG["IMG_SIZE"],
    channels=CONFIG["CHANNELS"]
)

dataloader = DataLoader(
    dataset,
    batch_size=CONFIG["BATCH_SIZE"],
    shuffle=True,
    num_workers=CONFIG["NUM_WORKERS"],
    pin_memory=(device.type == "cuda"),
    drop_last=True
)

print("Dataset size:", len(dataset))
print("Batches per epoch:", len(dataloader))

## Step 9 — Define the Generator and Discriminator

This is a **DCGAN-style GAN**.

### Generator
Turns random noise into a synthetic floorplan image.

### Discriminator
Looks at an image and predicts whether it is:
- real (from the dataset)
- fake (made by the generator)


In [ ]:
def weights_init(module):
    classname = module.__class__.__name__
    if classname.find("Conv") != -1:
        nn.init.normal_(module.weight.data, 0.0, 0.02)
    elif classname.find("BatchNorm") != -1:
        nn.init.normal_(module.weight.data, 1.0, 0.02)
        nn.init.constant_(module.bias.data, 0)


class Generator(nn.Module):
    def __init__(self, latent_dim=100, channels=1, features=64):
        super().__init__()
        self.net = nn.Sequential(
            # Input: (latent_dim, 1, 1)
            nn.ConvTranspose2d(latent_dim, features * 8, kernel_size=4, stride=1, padding=0, bias=False),
            nn.BatchNorm2d(features * 8),
            nn.ReLU(True),

            # 4x4 -> 8x8
            nn.ConvTranspose2d(features * 8, features * 4, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(features * 4),
            nn.ReLU(True),

            # 8x8 -> 16x16
            nn.ConvTranspose2d(features * 4, features * 2, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(features * 2),
            nn.ReLU(True),

            # 16x16 -> 32x32
            nn.ConvTranspose2d(features * 2, features, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(features),
            nn.ReLU(True),

            # 32x32 -> 64x64
            nn.ConvTranspose2d(features, channels, kernel_size=4, stride=2, padding=1, bias=False),
            nn.Tanh()
        )

    def forward(self, z):
        return self.net(z)


class Discriminator(nn.Module):
    def __init__(self, channels=1, features=64):
        super().__init__()
        self.net = nn.Sequential(
            # 64x64 -> 32x32
            nn.Conv2d(channels, features, kernel_size=4, stride=2, padding=1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),

            # 32x32 -> 16x16
            nn.Conv2d(features, features * 2, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(features * 2),
            nn.LeakyReLU(0.2, inplace=True),

            # 16x16 -> 8x8
            nn.Conv2d(features * 2, features * 4, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(features * 4),
            nn.LeakyReLU(0.2, inplace=True),

            # 8x8 -> 4x4
            nn.Conv2d(features * 4, features * 8, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(features * 8),
            nn.LeakyReLU(0.2, inplace=True),

            # 4x4 -> 1x1
            nn.Conv2d(features * 8, 1, kernel_size=4, stride=1, padding=0, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x).view(-1, 1)

G = Generator(
    latent_dim=CONFIG["LATENT_DIM"],
    channels=CONFIG["CHANNELS"],
    features=CONFIG["GEN_FEATURES"]
).to(device)

D = Discriminator(
    channels=CONFIG["CHANNELS"],
    features=CONFIG["DISC_FEATURES"]
).to(device)

G.apply(weights_init)
D.apply(weights_init)

print("✅ Generator and Discriminator created.")
print("Generator params:", sum(p.numel() for p in G.parameters()))
print("Discriminator params:", sum(p.numel() for p in D.parameters()))

## Step 10 — Loss and optimizers


In [ ]:
criterion = nn.BCELoss()

optimizer_G = optim.Adam(
    G.parameters(),
    lr=CONFIG["LR"],
    betas=(CONFIG["BETA1"], 0.999)
)

optimizer_D = optim.Adam(
    D.parameters(),
    lr=CONFIG["LR"],
    betas=(CONFIG["BETA1"], 0.999)
)

fixed_noise = torch.randn(CONFIG["GENERATE_N"], CONFIG["LATENT_DIM"], 1, 1, device=device)

print("✅ Loss and optimizers ready.")

## Step 11 — Helper functions for visualization and saving


In [ ]:
def denormalize(tensor):
    return tensor * 0.5 + 0.5


def show_tensor_grid(tensor_batch, nrow=4, title="Image grid"):
    grid = make_grid(denormalize(tensor_batch), nrow=nrow)
    grid_np = grid.detach().cpu().permute(1, 2, 0).numpy()

    plt.figure(figsize=(10, 10))
    if CONFIG["CHANNELS"] == 1:
        plt.imshow(grid_np.squeeze(), cmap="gray")
    else:
        plt.imshow(grid_np)
    plt.title(title)
    plt.axis("off")
    plt.show()


def save_generated_grid(tensor_batch, file_path, nrow=4):
    save_image(denormalize(tensor_batch), file_path, nrow=nrow)


def save_individual_samples(tensor_batch, out_dir):
    Path(out_dir).mkdir(parents=True, exist_ok=True)
    for i, sample in enumerate(tensor_batch):
        save_image(denormalize(sample), f"{out_dir}/sample_{i+1:02d}.png")


def save_checkpoint(generator, discriminator, optimizer_G, optimizer_D, history, file_path):
    checkpoint = {
        "generator_state_dict": generator.state_dict(),
        "discriminator_state_dict": discriminator.state_dict(),
        "optimizer_G_state_dict": optimizer_G.state_dict(),
        "optimizer_D_state_dict": optimizer_D.state_dict(),
        "history": history,
        "config": CONFIG
    }
    torch.save(checkpoint, file_path)
    print("✅ Checkpoint saved to:", file_path)

## Step 12 — Train the GAN

This is the main training loop.

### What happens in each batch
1. Train the **Discriminator**
   - real floorplans should get label 1
   - fake floorplans should get label 0

2. Train the **Generator**
   - tries to fool the discriminator into predicting 1 for fake images

### Beginner note
GAN losses can be noisy. That is normal.
The generated image previews matter more than perfectly smooth loss curves.


In [ ]:
g_losses = []
d_losses = []
preview_snapshots = []

real_label_value = 0.9   # slight label smoothing
fake_label_value = 0.0

for epoch in range(CONFIG["EPOCHS"]):
    running_g = 0.0
    running_d = 0.0

    pbar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{CONFIG['EPOCHS']}")
    for real_imgs in pbar:
        real_imgs = real_imgs.to(device, non_blocking=True)
        batch_size = real_imgs.size(0)

        # ---------------------------------
        # Train Discriminator
        # ---------------------------------
        D.zero_grad(set_to_none=True)

        real_labels = torch.full((batch_size, 1), real_label_value, device=device)
        fake_labels = torch.full((batch_size, 1), fake_label_value, device=device)

        output_real = D(real_imgs)
        loss_D_real = criterion(output_real, real_labels)

        noise = torch.randn(batch_size, CONFIG["LATENT_DIM"], 1, 1, device=device)
        fake_imgs = G(noise)

        output_fake = D(fake_imgs.detach())
        loss_D_fake = criterion(output_fake, fake_labels)

        loss_D = loss_D_real + loss_D_fake
        loss_D.backward()
        optimizer_D.step()

        # ---------------------------------
        # Train Generator
        # ---------------------------------
        G.zero_grad(set_to_none=True)

        target_labels_for_G = torch.full((batch_size, 1), 1.0, device=device)
        output_fake_for_G = D(fake_imgs)
        loss_G = criterion(output_fake_for_G, target_labels_for_G)

        loss_G.backward()
        optimizer_G.step()

        running_d += loss_D.item()
        running_g += loss_G.item()

        pbar.set_postfix({
            "loss_D": f"{loss_D.item():.4f}",
            "loss_G": f"{loss_G.item():.4f}"
        })

    avg_d = running_d / max(len(dataloader), 1)
    avg_g = running_g / max(len(dataloader), 1)

    d_losses.append(avg_d)
    g_losses.append(avg_g)

    print(f"Epoch {epoch+1}: avg_loss_D={avg_d:.4f}, avg_loss_G={avg_g:.4f}")

    if (epoch + 1) % CONFIG["PREVIEW_EVERY"] == 0 or (epoch + 1) == CONFIG["EPOCHS"]:
        with torch.no_grad():
            preview = G(fixed_noise).detach().cpu()
        preview_snapshots.append((epoch + 1, preview))
        show_tensor_grid(preview, nrow=4, title=f"Generated samples after epoch {epoch+1}")

print("✅ Training complete.")

## Step 13 — Plot Generator and Discriminator losses


In [ ]:
plt.figure(figsize=(9, 4))
plt.plot(range(1, len(g_losses) + 1), g_losses, label="Generator loss")
plt.plot(range(1, len(d_losses) + 1), d_losses, label="Discriminator loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("GAN Training Loss")
plt.legend()
plt.grid(True)
plt.show()

## Step 14 — Save the trained checkpoint


In [ ]:
checkpoint_path = f"{CONFIG['OUTPUT_DIR']}/task12_gan_floorplans_checkpoint.pth"
save_checkpoint(G, D, optimizer_G, optimizer_D, {"g_losses": g_losses, "d_losses": d_losses}, checkpoint_path)

## Step 15 — Generate final synthetic floorplans


In [ ]:
with torch.no_grad():
    final_noise = torch.randn(CONFIG["GENERATE_N"], CONFIG["LATENT_DIM"], 1, 1, device=device)
    generated_samples = G(final_noise).detach().cpu()

show_tensor_grid(generated_samples, nrow=4, title="Final Synthetic Floorplans")

## Step 16 — Save generated images


In [ ]:
grid_path = f"{CONFIG['OUTPUT_DIR']}/synthetic_floorplans_grid.png"
single_dir = f"{CONFIG['OUTPUT_DIR']}/synthetic_floorplans"

save_generated_grid(generated_samples, grid_path, nrow=4)
save_individual_samples(generated_samples, single_dir)

print("✅ Saved generated outputs:")
print("-", grid_path)
print("-", single_dir)

## Step 17 — Save loss values to CSV


In [ ]:
import csv

loss_csv_path = f"{CONFIG['OUTPUT_DIR']}/gan_training_loss.csv"
with open(loss_csv_path, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["epoch", "generator_loss", "discriminator_loss"])
    for i, (g, d) in enumerate(zip(g_losses, d_losses), start=1):
        writer.writerow([i, g, d])

print("✅ Saved loss history to:", loss_csv_path)

## Step 18 — Create a short summary text file


In [ ]:
summary_text = f"""
Task 12 Summary

Dataset:
- Number of floorplan images used: {len(dataset)}
- Image size: {CONFIG['IMG_SIZE']} x {CONFIG['IMG_SIZE']}
- Channels: {CONFIG['CHANNELS']}

GAN Training:
- Epochs: {CONFIG['EPOCHS']}
- Batch size: {CONFIG['BATCH_SIZE']}
- Learning rate: {CONFIG['LR']}
- Final generator loss: {g_losses[-1] if len(g_losses) > 0 else 'N/A'}
- Final discriminator loss: {d_losses[-1] if len(d_losses) > 0 else 'N/A'}

Generation:
- Number of synthetic floorplans generated: {CONFIG['GENERATE_N']}

Main output files:
- {checkpoint_path}
- {grid_path}
- {loss_csv_path}
""".strip()

summary_path = f"{CONFIG['OUTPUT_DIR']}/task12_summary.txt"
with open(summary_path, "w", encoding="utf-8") as f:
    f.write(summary_text)

print(summary_text)
print("\n✅ Saved summary to:", summary_path)

## Step 19 — Zip the outputs folder


In [ ]:
archive_base = "task12_outputs_archive"
archive_path = shutil.make_archive(archive_base, "zip", CONFIG["OUTPUT_DIR"])
print("✅ Created archive:", archive_path)

## Step 20 — Download the outputs


In [ ]:
from google.colab import files

files.download(checkpoint_path)
files.download(grid_path)
files.download(loss_csv_path)
files.download(summary_path)
files.download(archive_path)

## Step 21 — Optional: load the checkpoint later and generate again


In [ ]:
def load_checkpoint_and_generate(checkpoint_path, n_samples=8):
    if not os.path.exists(checkpoint_path):
        raise FileNotFoundError(f"Checkpoint not found: {checkpoint_path}")

    ckpt = torch.load(checkpoint_path, map_location=device)

    loaded_G = Generator(
        latent_dim=ckpt["config"]["LATENT_DIM"],
        channels=ckpt["config"]["CHANNELS"],
        features=ckpt["config"]["GEN_FEATURES"]
    ).to(device)

    loaded_G.load_state_dict(ckpt["generator_state_dict"])
    loaded_G.eval()

    noise = torch.randn(n_samples, ckpt["config"]["LATENT_DIM"], 1, 1, device=device)
    with torch.no_grad():
        samples = loaded_G(noise).detach().cpu()

    show_tensor_grid(samples, nrow=4, title="Samples from loaded checkpoint")
    return samples

# Example:
# samples = load_checkpoint_and_generate(checkpoint_path, n_samples=8)

## Troubleshooting

### If training is too slow
Use this smaller setup in the config cell:
```python
CONFIG["BATCH_SIZE"] = 16
CONFIG["EPOCHS"] = 8
CONFIG["GEN_FEATURES"] = 32
CONFIG["DISC_FEATURES"] = 32
CONFIG["MAX_IMAGES"] = 1000
```

### If Colab runs out of memory
Reduce:
```python
CONFIG["BATCH_SIZE"] = 8
CONFIG["GEN_FEATURES"] = 32
CONFIG["DISC_FEATURES"] = 32
```

### If the outputs look noisy
That is normal for GANs, especially early in training.
Try:
- more epochs
- more images
- lower learning rate only if training becomes unstable

### If no images are found
Check the zip structure after extraction and make sure it contains image files with one of these extensions:
- `.png`
- `.jpg`
- `.jpeg`
- `.bmp`
- `.webp`

### If you want RGB instead of grayscale
Change:
```python
CONFIG["CHANNELS"] = 3
```
Then rerun the notebook from the config cell onward.
